# Instalations and Imports

<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/src/Topic_Clustring_and_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    # Running inside Google Colab → use !pip
    import google.colab
    in_google_colab = True
    print("Running in Colab → installing packages...")
    !pip install -q bertopic
    !pip install -q tweet-preprocessor
    !pip install -q spacy-entity-linker
    !pip install -q emoji
    !pip install sentence-transformers scikit-learn
    !pip install "transformers>=4.40.0" sentencepiece accelerate torch
    !pip install langdetect
except ImportError:
    # Not running in Colab
    in_google_colab = False
    pass

In [ ]:
import pandas as pd
import pyarrow  # ensures pandas can read/write parquet
import matplotlib.pyplot as plt
import numpy as np

import re
import emoji

from sentence_transformers import SentenceTransformer

from tqdm.auto import tqdm
from IPython.display import display
from langdetect import detect

In [ ]:
min_topic_size = 256 # BERT

# Loading and Exporting
[Labeled Datasets for Research on Information Operations](https://zenodo.org/records/14189193)

## Load Data

In [ ]:
dataset_name = "Labeled_Datasets/Ecuador"  # folder path
file_name = "Ecuador_part_all.gzip.parquet"  # file name

# if run local: put data in "../data/<dataset_name>/<file_name>"
# Recommended: working directory is: "Backbone-Graph/src"

In [ ]:
from pathlib import Path

# Google Colab
if in_google_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_path = Path("/content/drive/Shareddrives/KeyBoostAuthor") / dataset_name / file_name
    print("Running inside Colab. Loading from Google Drive...")

# Cluster\Local
else:
    # Recommended: working directory is: "Backbone-Graph/src"
    dataset_path = Path("../data") / dataset_name / file_name
    print(f"Running on local/cluster.\nLoading data from: {dataset_path}")


# Load the parquet file
df = pd.read_parquet(dataset_path)

# for testing
# df = df.sample(n=20, random_state=42).copy()

## Output Path


In [ ]:
from datetime import datetime
today = datetime.now().strftime("%Y_%m_%d")


dataset_path = Path(dataset_path)


if in_google_colab:
  output_folder_path = (dataset_path.parents[2] / "Labeled_Dataset_with_Topic_NER" / dataset_path.parent.name / dataset_path.name)
else:
  # working directory is: Backbone-Graph/src
  file_name_no_suffixes = file_name.split(".")[0]
  # output_folder_path = Path("../results") / dataset_name / file_name_no_suffixes / today
  output_folder_path = Path("../final_results") / dataset_name / file_name_no_suffixes / f"BERT_min_topic_size_{min_topic_size}" # final results
  # Example: ""../results/Labeled_Datasets/Ecuador/Ecuador_part_4/2025_11_27"



output_folder_path.mkdir(parents=True, exist_ok=True) # Createst the output directory if needed

print(f"output_folder_path = {output_folder_path}")

In [ ]:
df.head()

## Logs

In [ ]:
from functools import wraps
from datetime import datetime



REPORT_PATH = output_folder_path / "progress_report.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"

    print(line)
    with open(REPORT_PATH, "a") as f:
        f.write(line + "\n")


def report_done(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as e:
            write_report(f"FAILED: {func.__name__} | {e}")
            write_report("#######################################################")
            write_report("#######################################################")
            raise
    return wrapper

## Save df

In [ ]:
def save_df(df, output_folder_path, file_name):
    df_output_path = output_folder_path / f"processed_{file_name}"
    df.to_parquet(df_output_path, index=False, compression="gzip")
    write_report(f"df is saved to: {df_output_path}")

# Dataset Statistics

In [ ]:
total_posts = len(df)
hashtag_posts = df[df['hashtags'].notna() & (df['hashtags'].str.len() > 0)]
mention_posts = df[df['account_mentions'].notna() & (df['account_mentions'].str.len() > 0)]
url_posts = df[df['urls'].notna() & (df['urls'].str.len() > 0)]
start_date = df["post_time"].min().strftime("%Y-%m")
end_date = df["post_time"].max().strftime("%Y-%m")

stats_text = [
    f"Posts time range: {start_date} → {end_date}",
    "",
    f"{'df shape:':<40} {df.shape}",
    f"{'df posts with hashtags # shape:':<40} {hashtag_posts.shape}  ({len(hashtag_posts)/total_posts*100:.2f}%)",
    f"{'df posts with account_mentions @ shape:':<40} {mention_posts.shape}  ({len(mention_posts)/total_posts*100:.2f}%)",
    f"{'df posts with urls shape:':<40} {url_posts.shape}  ({len(url_posts)/total_posts*100:.2f}%)",
]

# print to console
print("\n".join(stats_text))

# save to file
if output_folder_path:
    output_folder_path = Path(output_folder_path)
    report_path = output_folder_path / "dataset_summary.txt"
    with open(report_path, "w") as f:
        f.write("\n".join(stats_text))

print()
print()

print(f"{'**collum name**':<30} **value type**")
for col in df.columns:
    # dtype_str = str(df[col].dtype)  # Convert dtype to string
    # get a sample non-null value if exists
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")


print()
print()
print(df['post_language'].value_counts())

### Postid Duplication

In [ ]:
postid_duplication_sum = df["postid"].duplicated().sum()
write_report(
    f"postid duplication sum {postid_duplication_sum} "
    f"({postid_duplication_sum / len(df) * 100:.3f}%)"
)

# Clean Text

## Clean text With Entities

In [ ]:
def clean_url(url_match):

    url = url_match.group(0)

    # 1. Remove protocol (http:// or https://)
    url = re.sub(r"https?://", "", url)
    url = re.sub(r"www\.", "", url)

    # 2. Remove query parameters (?x=1)
    url = url.split("?")[0]

    # 3. Split into components by "/"
    parts = url.split("/")

    # Domain (first part)
    domain = parts[0].replace(".", "_")  # twitter.com → twitter_com

    # First section (second part of the URL)
    if len(parts) > 1 and parts[1].strip() != "":
        section = parts[1].replace(".", "_")
        token = f" URL_{domain}_{section} "
    else:
        token = f" URL_{domain} "

    return token

In [ ]:
def clean_text(text: str) -> str:

    # lowercase
    text = str(text).lower()

    # URLs → domian + first section
    text = re.sub(r"(https?://\S+|www\.\S+)", clean_url, text)

    # Hashtags → HASHTAG_xxx
    text = re.sub(r"#(\w+)", r" HASHTAG_\1 ", text)

    # Mentions → MENTION_xxx
    text = re.sub(r"@(\w+)", r" MENTION_\1 ", text)

    # emoji → EMOJI_xxx
    text = emoji.demojize(text, delimiters=(" EMOJI_", " "))

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["clean_text_with_entities"] = df["post_text"].apply(clean_text)

## Clean Text Without Enttities

In [ ]:
import re
import pandas as pd

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
WHITESPACE_RE = re.compile(r"\s+")

def clean_enteties(text: str) -> str:
    """
    Lightly clean a social-media post while keeping it natural
    for machine translation.

    Operations:
    - Replace URLs with <URL>
    - Replace @username with <USER>
    - Turn #hashtag into 'hashtag' (drop the #)
    - Collapse repeated whitespace
    """
    if not isinstance(text, str):
        text = str(text)

    # lowercase
    text = str(text).lower()

    # 1. Normalize URLs
    text = URL_RE.sub("<URL>", text)

    # 2. Normalize mentions
    text = MENTION_RE.sub("<USER>", text)

    # 3. Normalize hashtags: "#Ecuador2025" -> "Ecuador2025"
    text = HASHTAG_RE.sub(r"\1", text)

    # 4. Collapse whitespace
    text = WHITESPACE_RE.sub(" ", text).strip()

    return text

In [ ]:
df["clean_text_without_enteties"] = df["post_text"].astype(str).apply(clean_enteties)

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

df[["post_text", "clean_text_without_enteties"]].sample(n=20)

# Translation to English

## Language Codes 

In [ ]:
# map post_language codes to NLLB-200 language codes
# https://dl-translate.readthedocs.io/en/latest/available_languages/

def map_to_nllb_code(lang_code: str, text: str, undefined_code = "und") -> str:
    """
    Map your dataset's post_language codes to NLLB-200 language codes.
    If not in map, auto-detect the language from text.
    Args:
        lang_code (str): Post language code.
        text (str): Post text for detection.
        undefined_code (str): Default code to return if no mapping is found.
    Returns:
        str: NLLB-200 language code.
    """
    nllb_mapping = {

    # 🌍 Africa
    "af": "afr_Latn",   # Afrikaans
    "am": "amh_Ethi",   # Amharic
    "sw": "swh_Latn",   # Swahili
    "so": "som_Latn",   # Somali

    # 🌍 Middle East
    "ar": "arb_Arab",   # Arabic (MSA)
    "he": "heb_Hebr",   # Hebrew
    "fa": "pes_Arab",   # Persian (Farsi)
    "ur": "urd_Arab",   # Urdu
    "tr": "tur_Latn",   # Turkish

    # 🌍 Europe
    "be": "bel_Cyrl",   # Belarusian
    "bg": "bul_Cyrl",   # Bulgarian
    "bs": "bos_Latn",   # Bosnian
    "ca": "cat_Latn",   # Catalan
    "cs": "ces_Latn",   # Czech
    "cy": "cym_Latn",   # Welsh
    "da": "dan_Latn",   # Danish
    "de": "deu_Latn",   # German
    "el": "ell_Grek",   # Greek
    "en": "eng_Latn",   # English
    "es": "spa_Latn",   # Spanish
    "et": "est_Latn",   # Estonian
    "fi": "fin_Latn",   # Finnish
    "fr": "fra_Latn",   # French
    "ga": "gle_Latn",   # Irish
    "gl": "glg_Latn",   # Galician
    "hr": "hrv_Latn",   # Croatian
    "hu": "hun_Latn",   # Hungarian
    "is": "isl_Latn",   # Icelandic
    "it": "ita_Latn",   # Italian
    "lt": "lit_Latn",   # Lithuanian
    "lv": "lvs_Latn",   # Latvian
    "mk": "mkd_Cyrl",   # Macedonian
    "mt": "mlt_Latn",   # Maltese
    "nl": "nld_Latn",   # Dutch
    "no": "nob_Latn",   # Norwegian (Bokmål)
    "pl": "pol_Latn",   # Polish
    "pt": "por_Latn",   # Portuguese
    "ro": "ron_Latn",   # Romanian
    "ru": "rus_Cyrl",   # Russian
    "sk": "slk_Latn",   # Slovak
    "sl": "slv_Latn",   # Slovenian
    "sq": "als_Latn",   # Albanian
    "sr": "srp_Cyrl",   # Serbian (Cyrillic default)
    "sv": "swe_Latn",   # Swedish
    "uk": "ukr_Cyrl",   # Ukrainian

    # 🌍 Caucasus & Central Asia
    "az": "azj_Latn",   # Azerbaijani
    "hy": "hye_Armn",   # Armenian
    "ka": "kat_Geor",   # Georgian
    "kk": "kaz_Cyrl",   # Kazakh
    "mn": "mon_Cyrl",   # Mongolian
    "uz": "uzn_Latn",   # Uzbek

    # 🌍 South Asia
    "bn": "ben_Beng",   # Bengali
    "gu": "guj_Gujr",   # Gujarati
    "hi": "hin_Deva",   # Hindi
    "kn": "kan_Knda",   # Kannada
    "ml": "mal_Mlym",   # Malayalam
    "mr": "mar_Deva",   # Marathi
    "ne": "npi_Deva",   # Nepali
    "pa": "pan_Guru",   # Punjabi
    "ta": "tam_Taml",   # Tamil
    "te": "tel_Telu",   # Telugu

    # 🌍 Southeast Asia
    "id": "ind_Latn",   # Indonesian
    "ms": "zsm_Latn",   # Malay
    "th": "tha_Thai",   # Thai
    "tl": "tgl_Latn",   # Tagalog
    "vi": "vie_Latn",   # Vietnamese
    "km": "khm_Khmr",   # Khmer

    # 🌍 East Asia
    "ja": "jpn_Jpan",   # Japanese
    "ko": "kor_Hang",   # Korean
    "zh": "zho_Hans",   # Chinese (Simplified)
}

    # 1. direct mapping
    if lang_code in nllb_mapping:
        return nllb_mapping[lang_code]

    # 2. language detection
    try:
        detected = detect(text)
    except Exception as e:
        write_report(f"Language detection failed: {e} | lang_code={lang_code} | text={text[:100]}")
        return undefined_code  # undetermined

    # 3. mapping detected language
    if detected in nllb_mapping:
        return nllb_mapping[detected]

    write_report(f"No NLLB mapping for detected language: {detected}")
    return lang_code

In [ ]:
# Change post_language to NLLB-200 codes
undefined_code = "und"

# Before mapping
print("Before mapping to NLLB-200 codes:")
language_value_counts = df["post_language"].value_counts()
print(language_value_counts)
write_report(f"Undefined post language percentage **before** mapping: {language_value_counts.get('und', 0) / len(df) * 100:.2f}%")

# Apply mapping
df["post_language"] = df.apply(lambda row: map_to_nllb_code(row['post_language'], row['clean_text_without_enteties'], undefined_code), axis=1)

# After mapping to NLLB-200 codes:
print("After mapping to NLLB-200 codes:")
language_value_counts = df["post_language"].value_counts()
print(language_value_counts)
write_report(f"Undefined post language percentage **after** mapping: {language_value_counts.get('und', 0) / len(df) * 100:.2f}%")

In [ ]:
# Remove post with undefined language
undefined_count = df[df['post_language'] == undefined_code].shape[0]
write_report(f"Removing {undefined_count} posts with undefined language code {undefined_code}")
df = df[df['post_language'] != undefined_code].copy()

## Translation

In [ ]:
# load models

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

MODEL_NAME = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"

write_report(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
write_report(f"Loaded {MODEL_NAME} model and tokenizer on {device}")

translation_pipeline = pipeline("translation", model=model, tokenizer=tokenizer, device=device, torch_dtype=torch.float16, max_length=512, batch_size=8)
write_report(f"Translation pipeline created.")

In [ ]:
import torch 

def _chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n], i

@report_done
def transformers_translator(
    df: pd.DataFrame,
    translation_pipe,
    text_col: str = "clean_text_without_enteties",
    lang_col: str = "post_language",
    out_col: str = "translated_post",
    target_lang: str = "eng_Latn",
    pivot_lang: str = "spa_Latn",
    backtranslate_same_lang: bool = True,
    call_batch: int = 16,          # how many texts you feed per call (NOT pipeline batch_size)
    max_new_tokens: int = 512,      # cap generated length (huge VRAM saver)
) -> pd.DataFrame:

    result_df = pd.DataFrame(index=df.index)
    result_df[out_col] = None

    for src_lang, group in tqdm(df.groupby(lang_col), desc="Translating by language"):
        texts = group[text_col].astype(str).tolist()
        indexes = group.index.to_list()

        translated_all = [None] * len(texts)

        def run_translate(batch_texts, s_lang, t_lang):
            # inference_mode saves memory vs no_grad in many cases
            with torch.inference_mode():
                outs = translation_pipe(
                    batch_texts,
                    src_lang=s_lang,
                    tgt_lang=t_lang,
                    # truncation=True,                # avoid super-long inputs
                    max_new_tokens=max_new_tokens,  # avoid 1024-token generations
                )
            return [o["translation_text"] for o in outs]

        # Case 1: normal translation
        if src_lang != target_lang:
            write_report(f"{len(texts)} posts: {src_lang} → {target_lang}")

            for batch_texts, start in _chunks(texts, call_batch):
                batch_trans = run_translate(batch_texts, src_lang, target_lang)
                translated_all[start:start+len(batch_trans)] = batch_trans

                # cleanup between calls (helps on long runs)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        # Case 2: back-translation
        elif backtranslate_same_lang:
            write_report(f"{len(texts)} posts: {target_lang} → {pivot_lang} → {target_lang}")

            for batch_texts, start in _chunks(texts, call_batch):
                pivot_texts = run_translate(batch_texts, target_lang, pivot_lang)
                back_texts = run_translate(pivot_texts, pivot_lang, target_lang)
                translated_all[start:start+len(back_texts)] = back_texts

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        # assign back
        result_df.loc[indexes, out_col] = translated_all

    return result_df

In [ ]:
print(f"df shape: {df.shape}")
translated_df = transformers_translator(df, translation_pipeline)
df = df.drop(columns=["translated_post"], errors="ignore") # Drop the old column if exist
df = df.join(translated_df)
write_report("Translation completed.")

In [ ]:
pd.set_option("display.width", 3000)
pd.set_option("display.max_colwidth", 1500)

df.sample(10)[["clean_text_without_enteties", "translated_post"]]

## Remove Engllish Stopwords

In [ ]:
from nltk.corpus import stopwords
import nltk

In [ ]:
nltk.download('stopwords')

EXTRA_STOPWORDS = [
    'he', 'she', 'they', 'you', 'i', 'we', 'im', 'hes', 'shes', 'its', 'rt', 'amp', 're',
    'theyre', 'dont', 'doesnt', 'would', 'could', 'like', 'know', 'say', 'said', 'one',
    'look', 'looks', 'think', 'oh', 'yeah', 'guy', 'gonna', 'thing', 'someone', 'people',
    'get', 'got', 'thats', 'just', 'really', 'even', 'still', 'want', 'need', 'needs',
    'me', 'to', 'you', 'the', 'that', 'be', 'my', 'get', 'in', 'and',
     'rt', 'the', 'https', 'to', 'my', 'it', 'on', 'you', 'of', 'for', 'https', 'http', 'RT'
]
all_stopwords_english = set(stopwords.words('english') + EXTRA_STOPWORDS)

def remove_stopwords(text):
    if not text:
        return text
    return " ".join([w for w in text.lower().split() if w not in all_stopwords_english])


In [ ]:
df["clean_text_with_entities"] = df["clean_text_with_entities"].apply(remove_stopwords)
df["clean_text_without_enteties"] = df["clean_text_without_enteties"].apply(remove_stopwords)
df["translated_post"] = df["translated_post"].apply(remove_stopwords)

In [ ]:
df[["post_text", "clean_text_with_entities"]].head(20)

## Check Translation

In [ ]:
print("Untranslated posts samples:")
display(df[df["translated_post"].isna()][["post_text", "post_language", "clean_text_without_enteties", "translated_post"]])

# Check if more than 10% of posts are untranslated
num_untranslated = df["translated_post"].isna().sum()
total_posts = len(df)
percentage_untranslated = (num_untranslated / total_posts) * 100

write_report(f"Untranslated posts: {num_untranslated}/{total_posts} ({percentage_untranslated:.2f}%)")
if percentage_untranslated > 10:
    write_report(f"Handling None Cells: More than 10% of posts are untranslated: {percentage_untranslated:.2f}%")
    raise ValueError(f"More than 10% of posts are untranslated: {percentage_untranslated:.2f}%")

# Fill untranslated posts with original cleaned text
df["translated_post"] = df["translated_post"].fillna(df["clean_text_without_enteties"])
write_report("Filled untranslated posts with original cleaned text.")

## Save df

In [ ]:
save_df(df, output_folder_path, file_name)

write_report(
    "Finished Translation\n"
    f"Saved df to {output_folder_path}\n"
    "Continuing to Entities Reuse"
)

# Named Entity Recognition

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load model ONCE
tokenizer = AutoTokenizer.from_pretrained("TweebankNLP/bertweet-tb2_wnut17-ner")
ner_model = AutoModelForTokenClassification.from_pretrained("TweebankNLP/bertweet-tb2_wnut17-ner")

ner_pipeline = pipeline(
    "token-classification",
    model=ner_model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

# @report_done
def extract_ner_entities(posts_lst: list):
    """
    Runs NER in batches on a list of text posts.

    Args:
          posts_lst (list): List of strings (cleaned posts).
    Returns
    -------
    all_entities (list[list[str]]): For each post: a list of extracted entity words.
    """

    ner_outputs = ner_pipeline(posts_lst)
    all_entities = []
    for post_output in ner_outputs:
        words = [ent["word"] for ent in post_output]
        all_entities.append(words)

    return all_entities

In [ ]:
# Apply ONCE, no pandas apply, in order to utilize batches.
df["ner_entities"] = extract_ner_entities(df["translated_post"].to_list())

In [ ]:
# To Delete
df_path = "/home/amitner/Backbone-Graph/results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/processed_Ecuador_part_all.gzip.parquet"
df = pd.read_parquet(df_path)
df["ner_entities"] = extract_ner_entities(df["translated_post"].to_list())
df.to_parquet(df_path, index=False, compression="gzip")

In [ ]:
# display post with ner entities
df.loc[df["ner_entities"].astype(bool), ["post_text", "translated_post", "ner_entities"]].head()

# Entities Reuse

In [3]:
# Parameters
# ENTITY_COLS = ["hashtags", "urls", "account_mentions", "ner_entities"]
ENTITY_COLS = ["hashtags"]
# time_windows = ["1W", "4D", "12H"] # See pandas.TimeDelta documentation. Example: "1W" for 1 week, "4D" for 4 days, "12H" for 12 hours, etc.
time_windows = ["4D"] # For testing

In [4]:
from collections import defaultdict, deque
import numpy as np
import pandas as pd
from tqdm import tqdm


def add_entity_reuse(
    df: pd.DataFrame,
    time_window: str = "4D",
    entity_column: str = "hashtags",
    post_id_column: str = "postid",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
) -> None:
    """
    Adds temporal entity-reuse columns to a DataFrame.

    For each post, this function finds earlier posts within a `time_window` that share at least one entity.
    It adds columns with the post IDs and account IDs of these related posts.

    It maintains per-entity sliding windows to efficiently track recent usage,
    collects matching prior posts, and stores their post IDs and account IDs in new columns.
    It use dict{entity: deque[(timestamp, post_index)]} to track recent posts for each entity, expiring old entries as it goes.

    Note 1: An authors can reuse entities from their own earlier posts, and these will be included in the results. You can later set to ignore self-reuse when building the graph.
    Note 2: The function does not count for a post that reuses the same entity type multiple times between posts. It will only be counted once per post.

    Args:
        df (pd.DataFrame): The input DataFrame.
        entity_column (str): The column with entities for each post.
        time_window (str): The sliding window duration (default: "4D").
        post_id_column (str): The column with post identifiers (default: "postid").
        account_id_column (str): The column with account identifiers (default: "accountid").
        timestamp_column (str): The column with post timestamps (default: "post_time").

    Returns:
        None: Modifies the DataFrame in-place, adding:
              - f"{entity_column}_reuse_{time_window}_postids"
              - f"{entity_column}_reuse_{time_window}_accountids"
    """


    print(f"\n[add_entity_reuse] Adding entity reuse columns for '{entity_column}' with time window {time_window}...") # TOOD: change to write_report

    # 1. Define output column names
    output_post_ids_column = f"{entity_column}_reuse_{time_window}_postids"
    output_account_ids_column  = f"{entity_column}_reuse_{time_window}_accountids"

    # 2. Convert columns to NumPy arrays for performance
    timestamps = pd.to_datetime(df[timestamp_column]).to_numpy(dtype="datetime64[ns]")
    post_ids = df[post_id_column].to_numpy()
    account_ids = df[account_id_column].to_numpy()
    entities_per_post = df[entity_column].to_numpy()

    # Convert time window to numpy timedelta
    time_window_delta = np.timedelta64(pd.Timedelta(time_window).value, "ns")

    # Pre-allocate output lists
    num_posts = len(df)
    output_post_ids = [[] for _ in range(num_posts)]
    output_account_ids  = [[] for _ in range(num_posts)]

    # 3. Set up entity history tracker: {entity: deque[(timestamp, post_index)]}
    entities_history: dict[str, deque] = defaultdict(deque)

    # 4. Process posts in chronological order
    sorted_timestamp_indices = np.argsort(timestamps)
    for current_post_index in tqdm(sorted_timestamp_indices, desc="Processing posts"):
        current_timestamp = timestamps[current_post_index]
        current_entities = entities_per_post[current_post_index]

        # no entities in this post
        if current_entities is None or (len(current_entities) == 0):
            continue


        matching_post_indices = set()

        # 5. Find related earlier posts for each entity
        for entity in current_entities:
            
            # no entity
            if not entity:
                continue

            entity_post_deque = entities_history[entity] # deque: [(timestamp, post_index)] of post with this entity
            
            # Expire old posts from the deque
            while entity_post_deque and (current_timestamp - entity_post_deque[0][0]) > time_window_delta:
                entity_post_deque.popleft() # remove posts outside the time window

            # Collect indices of posts still in the window
            matching_post_indices.update(post_index for _, post_index in entity_post_deque)

        # 6. Store results in df
        if matching_post_indices:
            sorted_matching_indices = sorted(list(matching_post_indices), key=lambda index: timestamps[index]) # label order by timestamps, oldest is first

            # Remove duplicates while preserving order
            output_post_ids[current_post_index] = list(dict.fromkeys([post_ids[post_index] for post_index in sorted_matching_indices]))
            output_account_ids[current_post_index] = list(dict.fromkeys([account_ids[post_index] for post_index in sorted_matching_indices]))

        # 7. Add current post to history (after matching)
        for entity in current_entities:
            if entity:
                entities_history[entity].append((current_timestamp, current_post_index))

    # 8. Add results to DataFrame
    df[output_post_ids_column] = output_post_ids
    print(f"[add_entity_reuse] Added column: {output_post_ids_column}") # TOOD: change to write_report
    df[output_account_ids_column]  = output_account_ids
    print(f"[add_entity_reuse] Added column: {output_account_ids_column}") # TOOD: change to write_report


In [5]:
# TEMP CELL
temp_df_path = "../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/processed_Ecuador_part_all.gzip.parquet"

In [6]:
# TEMP CELL
df = pd.read_parquet(temp_df_path)

In [7]:
# EMP CELL
df = df.rename(
    columns=lambda c: c.replace("BERTopic_prob", "BERTopic_prob_")
)

In [8]:
for time_window in time_windows:
    print(f"\nProcessing time window: {time_window}")
    for entity_col in ENTITY_COLS:
        add_entity_reuse(df, time_window=time_window, entity_column=entity_col,)


Processing time window: 4D

[add_entity_reuse] Adding entity reuse columns for 'hashtags' with time window 4D...


Processing posts: 100%|██████████| 1468565/1468565 [08:32<00:00, 2865.75it/s] 


[add_entity_reuse] Added column: hashtags_reuse_4D_postids
[add_entity_reuse] Added column: hashtags_reuse_4D_accountids


In [9]:
# TEMP CELL
df.to_parquet(temp_df_path, index=False, compression="gzip")
print(f"Saved df with new entity reuse columns to: {temp_df_path}")

Saved df with new entity reuse columns to: ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/processed_Ecuador_part_all.gzip.parquet


In [10]:
# EMP CELL
print(f"df columns: {df.columns}")

df columns: Index(['postid', 'post_text', 'application_name', 'post_language',
       'in_reply_to_postid', 'in_reply_to_accountid', 'post_time', 'accountid',
       'account_profile_description', 'follower_count', 'following_count',
       'account_creation_date', 'is_repost', 'reposted_accountid',
       'reposted_postid', 'hashtags', 'urls', 'account_mentions', 'is_control',
       'clean_text_with_entities', 'clean_text_without_enteties',
       'translated_post', 'BERTopic_topic_512', 'BERTopic_prob_512',
       'BERTopic_topic_256', 'BERTopic_prob_256', 'BERTopic_topic_128',
       'BERTopic_prob_128', 'ner_entities', 'hashtags_reuse_4D_postids',
       'hashtags_reuse_4D_accountids'],
      dtype='object')


In [14]:
display(
    df[df["hashtags_reuse_4D_postids"].str.len() > 0].head()
)

,postid,post_text,application_name,post_language,in_reply_to_postid,in_reply_to_accountid,post_time,accountid,account_profile_description,follower_count,...,translated_post,BERTopic_topic_512,BERTopic_prob_512,BERTopic_topic_256,BERTopic_prob_256,BERTopic_topic_128,BERTopic_prob_128,ner_entities,hashtags_reuse_4D_postids,hashtags_reuse_4D_accountids
47,d226f80ba748301024bc4d62eddffe4bc7e331863c,Que insolentes estos Paraguayos #par irrespeta...,62010408238d4a0f958ec97eec3db0a4f69bcd0d11,spa_Latn,None,None,2010-06-14 19:14:09,196bf752a3197efee3ca56e119495b427064243214,Turning ☕ into code. Senior Software Engineer ...,2851,...,insolent paraguayans disrespect italy.,0,0.597335,0,0.626642,2,0.777371,"[paraguay@@, italy@@]",[a9b5f9caa55fd5b203c2ae7eb1eb07af9b39cee034],[196bf752a3197efee3ca56e119495b427064243214]
51,d959cba124174998aa5b6675b4e8919ef9a03ed138,"%España, #spa, España, #spa, España% https://0...",2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,None,None,2010-06-14 20:28:00,86f64e2fce5d67c880f1d220f60205d26692ae9e6d,Abogada. Madre de dos príncipes.,135,...,"%spanish, spa, spa, spa, spa%spanish <url>",0,0.673808,0,0.668745,49,0.765399,"[span@@, spanish]",[589b2bfc232f6c92f512334f17c58845d1d52a6b7a],[86f64e2fce5d67c880f1d220f60205d26692ae9e6d]
55,1e05832acfb15c797dc3da7a07ac13e0e37485bde1,Bueno... mis equipos de #worldcup no juegan si...,62010408238d4a0f958ec97eec3db0a4f69bcd0d11,spa_Latn,None,None,2010-06-14 23:57:22,196bf752a3197efee3ca56e119495b427064243214,Turning ☕ into code. Senior Software Engineer ...,2851,...,well... world cup teams play wednesday spa fri...,1,0.695900,0,0.543750,3,0.696932,"[world cup, world cup]","[a9b5f9caa55fd5b203c2ae7eb1eb07af9b39cee034, d...",[196bf752a3197efee3ca56e119495b427064243214]
71,2b667945e1f71d0e83b87592ce78ef1b6e955ec79e,@f7b4bf6d3d3d3f424826d52d8058428d74466588fc: R...,a0cde693da15a23c629e21eb8da3cb9d170764524b,eng_Latn,None,None,2010-07-13 00:59:02,b97e4b56b7ce274e8ec30636172f0a723df7fe716e,IG: dasha__monae,396,...,high school confession.,0,0.534110,0,0.537179,74,0.620257,[],[2af42758e9197f1a0c385fb6b15e21b99184e457f3],[b97e4b56b7ce274e8ec30636172f0a723df7fe716e]
76,8c6baa360e0e42ddbe30b86ab99b9c291c37c6ec38,@eb8ca4555a5047982e5d26b9779e0f130e7ed16b55: #...,a0cde693da15a23c629e21eb8da3cb9d170764524b,eng_Latn,None,None,2010-07-13 02:17:11,b97e4b56b7ce274e8ec30636172f0a723df7fe716e,IG: dasha__monae,396,...,<user>: hsc girls & knew shit!! jealous whores...,0,0.536259,0,0.553797,1,0.576388,[],[7cf8da6696e7d60cccf01ed423605fd7064e129ee2],[b97e4b56b7ce274e8ec30636172f0a723df7fe716e]


## Validation

In [15]:
def sample_posts_reuse_entities(
    df: pd.DataFrame,
    reuse_col: str = "hashtags_reuse_4D_postids",
    random_state: int | None = None,
) -> pd.DataFrame:
    """
    Sample one post with non-empty reuse list and return it together
    with all reused posts.

    Args:
        df: Posts dataframe.
        reuse_col: Column containing list of reused postids.
        random_state: Optional seed for reproducibility.

    Returns:
        DataFrame containing:
            - The sampled post
            - All posts referenced in its reuse column
        With selected columns only.
    """

    cols_to_keep = [
        # "postid",
        #"accountid",
        "post_time",
        #"post_text",
        "translated_post",
        # "post_language",
        "hashtags",
        "account_mentions",
        "urls",
        "ner_entities",
        # reuse_col,
    ]

    # Filter non-empty reuse lists
    valid_rows = df[df[reuse_col].apply(lambda x: isinstance(x, list) and len(x) > 0)]

    if valid_rows.empty:
        raise ValueError(f"No posts found with non-empty {reuse_col}")

    # Sample one row
    sample_row = valid_rows.sample(1, random_state=random_state)

    reused_postids = sample_row[reuse_col].iloc[0]

    # Get reused posts
    reused_rows = df[df["postid"].isin(reused_postids)].sort_values("post_time")

    # Combine
    df_reuse_entities = pd.concat([sample_row, reused_rows])[cols_to_keep]

    return df_reuse_entities

In [17]:
reuse_col = "hashtags_reuse_4D_postids"
df_reuse_entities = sample_posts_reuse_entities(df, reuse_col=reuse_col)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", None
):
    entity_type = reuse_col.split("_reuse_")[0]
    print(f"The first post is the sampled post with reused {entity_type}, followed by the posts that the post use their {entity_type} within the {time_window}.")
    display(df_reuse_entities)

The first post is the sampled post with reused hashtags, followed by the posts that the post use their hashtags within the 4D.


,post_time,translated_post,hashtags,account_mentions,urls,ner_entities
381961,2017-01-24 23:55:27,"treasure nation, beautiful missuniverse thailand.","[MissUniverse, Thailand]",[6b9760fd682f5852d2d7763125ce1d1a75739858da],[],"[missuniverse, thail@@, and@@]"
380065,2017-01-23 03:39:37,dress first day show sugar playing wearing silk queen's craft centre designed joe surface missuniverse missuniversethailand url,"[MissUniverse, missuniversethailand]",[4e6ae911a14ca6a1b523029bdd29f10bd23940326c],[https://3df2fdae80a4d90ad0569e79ab80e8d8c7d6ad6461],"[queen's, craft, centre, joe, suniverse, missuniversethailand]"
380066,2017-01-23 03:40:16,"starting right now, national train arrived, hitting tag, missuniverse missthailand, helping us top 12.","[MissUniverse, missthailand]",[5ec3f6d5ce6b48e80c9e2d35295d76a9fe256580dc],[https://804e5b17ca4048f5301c826ec2afe555bd14ed846c],"[mis@@, suniverse, thailand@@]"
380076,2017-01-23 06:14:23,come can't come miss universe missuniversethailand misshailand chaling2mu <url>,"[MissUniverse, missuniversethailand, missthailand, chaling2mu]",[2595b4134b36857231972d264120e4dc539ad8c50c],[https://30cfd90ca2cbc0863f7c198e92717a81f6ecac3a9d],"[miss universe, missuniversethailand]"
380077,2017-01-23 06:14:32,philippines missuniverse missuniversethailand missthailand 65thmissuniverse missuniversethailand2016,"[MissUniverse, MissUniverseThailand, MissThailand, 65thMissUniverse, MissUniverseThailand2016]",[1f55bb0774badaedb9be1a628f74d0f503055d4036],[https://e9ee982d8395e7dd6fa8aa5b6fd9a20c6d21763523],"[philippines, thailand, thailand, ail@@]"
...,...,...,...,...,...,...
381956,2017-01-24 23:55:19,never serious tweeting missuniverse thailand.,"[MissUniverse, Thailand]",[28243922a4b6ee3e19e1358022b0694bc8c81fdd1b],[https://054968c62d368b7aaf6a1b32b6142696ecadf0027c],"[missuniverse, thail@@, and@@]"
381957,2017-01-24 23:55:21,"price... thailand, miss universe thailand?","[MissUniverse, Thailand]",[4a8117a9306a32badb894df36935def596aa753383],[],"[thailand@@, miss universe, thail@@, and@@]"
381958,2017-01-24 23:55:23,"train speed higher many countries, let's shy away it. train keeps going. train knows beautiful thai people.","[MissUniverse, Thailand]",[2ba18b75f2ff86e8758d4e500ec8d74abfa47fb328],[https://8bb17cc5987a670595fd1c617f66d85d66246e90b2],[thai]
381959,2017-01-24 23:55:24,"30th, hear hail throughout thailand, top 12 anything, miss universe thailand's third miss universe thailand.","[MissUniverse, Thailand]",[bafb59e45a79e9e8943041225160f425ce8379fe13],[],"[thailand@@, miss universe thail@@, and@@, miss universe thailand@@]"


## Save df

In [ ]:
save_df(df, output_folder_path, file_name)

write_report(
    "Finished Entities Reuse\n"
    "Continuing to Topic Clustering"
)

# English Topic Clustering

Note: the topic clustering doent influenced by the entities since it is preformed on the text without the entitites.

### Imports and Load df

In [ ]:
import pandas as pd
import numpy as np
import pyarrow  # ensures pandas can read/write parquet

from bertopic import BERTopic

import math
from typing import Iterator, List, Tuple
import gc

import matplotlib.pyplot as plt
from IPython.display import display

## Utils


### Posts Topic Stats

In [ ]:
@report_done
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        topic_stats (pd.DataFrame): Colums: [topic, total_count, io_Count, legitimate_posts, io_percent].
    """
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              total_count=(topic_col_name, "count"),
              io_count=("is_control", lambda x: (~x).sum())
          )
          .reset_index()
    )
    topic_stats["legitimate_posts"] = topic_stats["total_count"] - topic_stats["io_count"]
    topic_stats["io_percent"] = topic_stats["io_count"] / topic_stats["total_count"] * 100
    return topic_stats

In [ ]:
from matplotlib.lines import Line2D

def plot_topic_stats(topic_stats, topic_col_name, output_folder_path: str | Path = ""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.
        output_folder_path (str): Path to *folder* save the plot.

    Returns:
        None
    """
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["io_count"]
    legitimate_count = topic_stats["legitimate_posts"]
    io_percent = topic_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    bar_io = plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    bar_legit = plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    # Add IO percentage text
    ax = plt.gca()
    for x, io, legit, pct in zip(topics, io_count, legitimate_count, io_percent):
        total_height = io + legit
        ax.text(
            x,
            total_height + 0.5,              # slightly above the bar
            f"{pct:.1f}%",                 # format like 23.5%
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular"
        )

    plt.title(f"Post Count per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")

    percent_legend = Line2D([], [], color="none")
    total_posts_legend = Line2D([], [], color="none")
    plt.legend(
        handles=[bar_io, bar_legit, percent_legend, total_posts_legend],
        labels=[
            "IO posts",
            "Non-IO posts",
            "% IO Posts Ratio",
            f"Total posts: {topic_stats['total_count'].sum()}",
        ],
        loc="best",
    )


    plt.tight_layout()


    if output_folder_path:
        output_folder_path = Path(output_folder_path)
        plot_file_path = output_folder_path / (f"{topic_col_name}_posts_clustering.png")
        plt.savefig(plot_file_path, dpi=300)
        print(f"plot is saved to {plot_file_path}")

    plt.show()

    plt.close()

### Accoutns Topic Stats

In [ ]:
@report_done
def make_topic_account_stats(
    df: pd.DataFrame,
    topic_col: str,
    author_col: str = "accountid",
    is_control_col: str = "is_control",
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Compute per-topic unique account counts (IO vs control) using an author-level threshold.

    Args:
        df: Post-level DataFrame.
        topic_col: Topic column name.
        author_col: Author identifier column.
        is_control_col: Column where 1 = control, 0 = IO.
        threshold: IO-rate threshold to label an author as IO (1 = IO).

    Returns:
        DataFrame with per-topic counts of unique IO/control accounts and IO percentage.
    """
    # Author-level IO rate: 1 - mean(is_control)
    author_io_rate = 1 - df.groupby(author_col)[is_control_col].mean()
    author_io_tag = (author_io_rate >= threshold).astype(int)

    topic_author_df = df[[topic_col, author_col]].drop_duplicates().copy()
    topic_author_df["author_io_tag"] = topic_author_df[author_col].map(author_io_tag)

    stats = (
        topic_author_df
        .groupby(topic_col)["author_io_tag"]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Ensure both columns exist even if absent in data
    stats = stats.reindex(columns=[0, 1], fill_value=0).rename(
        columns={0: "control_accounts", 1: "io_accounts"}
    ).reset_index()

    stats["total_accounts"] = stats["control_accounts"] + stats["io_accounts"]
    stats["io_percent"] = (stats["io_accounts"] / stats["total_accounts"]).mul(100)

    return stats

In [ ]:
def plot_topic_account_stats(
    topic_account_stats: pd.DataFrame,
    topic_col_name: str,
    output_folder_path: Path | str | None = None,
):
    """Plot IO vs control account counts per topic.

    Args:
        topic_account_stats: Output of make_topic_account_stats().
        topic_col_name: Topic column name.
        model_name: Model name for title / filename.
        output_folder_path: Optional folder to save the plot.

    Returns:
        None
    """
    topics = topic_account_stats[topic_col_name]
    io_count = topic_account_stats["io_accounts"]
    control_count = topic_account_stats["control_accounts"]
    io_percent = topic_account_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO accounts
    bar_io = plt.bar(topics, io_count, label="IO accounts", color="red")

    # 🔵 Top: Control accounts
    bar_control = plt.bar(
        topics,
        control_count,
        bottom=io_count,
        label="Control accounts",
    )

    # Add IO percentage text
    ax = plt.gca()
    for x, io, ctrl, pct in zip(topics, io_count, control_count, io_percent):
        total = io + ctrl
        ax.text(
            x,
            total + 0.3,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular",
        )

    plt.title(f"Unique Accounts per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Unique Accounts")
    plt.xticks(topics)
    plt.grid(axis="y")

    # Legend with extra info lines
    percent_legend = Line2D([], [], color="none")
    total_accounts_legend = Line2D([], [], color="none")

    plt.legend(
        handles=[bar_io, bar_control, percent_legend, total_accounts_legend],
        labels=[
            "IO accounts",
            "Control accounts",
            "% IO Accounts Ratio",
            f"Total accounts: {topic_account_stats["total_accounts"].sum()}",
        ],
        loc="best",
    )

    if output_folder_path:
        output_folder_path = Path(output_folder_path)
        plot_file_path = output_folder_path / f"{topic_col_name}_accounts_clustering.png"
        plt.savefig(plot_file_path, dpi=300)
        print(f"Plot saved to {plot_file_path}")

    plt.show()
    plt.close()

### Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

post_text_col = "translated_post"

posts_texts = df[post_text_col].astype(str).tolist()

model = SentenceTransformer("digio/Twitter4SSE")

embeddings = model.encode(
    posts_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True, 
)

embeddings = embeddings.astype("float32", copy=False) # in order to use less memory
write_report(f"Generated embeddings shape: {embeddings.shape}")

## BERTopic

[BERTopic Parametes](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#calculate_probabilities)

In [ ]:
import subprocess
from bertopic import BERTopic
from umap import UMAP

@report_done
def english_BERTopic_modeling(
        posts_text_lst: list[str], 
        embeddings: np.ndarray, 
        output_folder_path: str | Path = "", 
        min_topic_size: int = 200,
    ):
    """
    Run BERTopic using precomputed embeddings.

    Args:
        posts_text_lst (list[str]): English text posts.
        embeddings (np.ndarray): Precomputed embeddings aligned with the posts.
        min_topic_size (int): Minimum cluster size.

    Returns:
        topics, probs, topic_model
    """

    # umap_model = UMAP(
    #     n_neighbors=15,
    #     n_components=5,
    #     min_dist=0.0,
    #     metric="cosine",
    #     low_memory=True,
    # )   

    topic_model = BERTopic(
        language="english",
        min_topic_size=min_topic_size,
        verbose=True,
        low_memory=True,
        # umap_model=umap_model,
    )   

    topics, probs = topic_model.fit_transform(posts_text_lst, embeddings)

    write_report(f"BERTopic modeling completed with min_topic_size={min_topic_size}.")
    write_report(f"BERTopic found {len(set(topics))} topics.")

    # save model to folder
    if output_folder_path:
        output_folder_path = Path(output_folder_path)
        topic_model.save(output_folder_path / "BERTopic_model")
        print(f"BERTopic model is saved to {output_folder_path}")

    return topics, probs, topic_model

### Run and Save

In [ ]:
BERTopic_topic, BERT_probs, BERTopic_model = english_BERTopic_modeling(posts_texts, embeddings, output_folder_path = output_folder_path, min_topic_size = min_topic_size)
df[f"BERTopic_topic"] = BERTopic_topic
df[f"BERTopic_prob"] = BERT_probs
write_report(f"Saved df with BERTopic topics. min_topic_size = {min_topic_size}")


save_df(df, output_folder_path, file_name)

### Graph and Stats

In [ ]:
BERTopic_model.get_topic_info()

In [ ]:
@report_done
def make_topic_top_posts_table(
    df,
    topic_col="BERTopic_topic",
    prob_col="BERTopic_prob",
    text_col="translated_post",
    top_k=10,
    output_folder_path: Path | str | None = None,
):
    """Make a df with the top k posts with the highest BERTopic probability for each topic."""

    top_posts = (
        df[[topic_col, prob_col, text_col]]
        .dropna(subset=[topic_col, prob_col, text_col])
        .sort_values([topic_col, prob_col], ascending=[True, False])
        .drop_duplicates(subset=[topic_col, text_col])  # remove duplicates within topic
        .groupby(topic_col)
        .head(top_k)
        .copy()
    )

    top_posts["rank"] = top_posts.groupby(topic_col).cumcount() + 1

    top_posts = top_posts.pivot(
        index=topic_col,
        columns="rank",
        values=text_col,
    )

    if output_folder_path is not None:
        output_folder_path = Path(output_folder_path)
        df_output_path = output_folder_path / "topics_top_posts.parquet"
        top_posts.to_parquet(df_output_path, compression="gzip")
        print(f"Topic posts table is saved to {df_output_path}")

    return top_posts

In [ ]:
topic_posts_table = make_topic_top_posts_table(df, output_folder_path = output_folder_path)
display(topic_posts_table.head())

In [ ]:
topic_col_name = "BERTopic_topic"
BERT_topic_stats = make_topic_stats(df, topic_col_name)
plot_topic_stats(BERT_topic_stats, topic_col_name, output_folder_path)

In [ ]:
topic_col_name = "BERTopic_topic"
topic_account_stats = make_topic_account_stats(df, topic_col_name)
display(topic_account_stats)
plot_topic_account_stats(topic_account_stats, topic_col_name, output_folder_path)

In [ ]:
try:
    fig = BERTopic_model.visualize_topics(custom_labels=True)
    display(fig)   # show it
except Exception as e:
    print("Not enough topics for visualization")
    print("Error:", e)

## K-Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

@report_done
def auto_kmeans_topics(embeddings, k_min=4, k_max=64, step = 4):
    best_k = None
    best_score = -1
    best_labels = None

    k_max =  min(k_max, len(embeddings)-1)

    for k in range(k_min, k_max + 1, step):
        print(f"Trying k={k}")
        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = km.fit_predict(embeddings)
        score = silhouette_score(embeddings, labels)

        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels


    print(f"\nBest k: {best_k}")
    return best_labels, best_k

In [ ]:
import math

# n = len(df)

# k_min = max(3,#  int(math.log(n)))
# print(f"k_min = {k_min}")
# k_max = min(200 ,int(math.sqrt(n)))
# print(f"k_max = {k_max}")
# labels, best_k = auto_kmeans_topics(embeddings, k_min=k_min, k_max=k_max)

labels, best_k = auto_kmeans_topics(embeddings, k_min=4, k_max=128, step=4)
df["K_Means_topic"] = labels
df.head()

In [ ]:
topic_col_name = "K_Means_topic"
topic_stats = make_topic_stats(df, topic_col_name)
plot_topic_stats(topic_stats, topic_col_name, output_folder_path)

In [ ]:
topic_col_name = "K_Means_topic"

topic_account_stats = make_topic_account_stats(df, topic_col_name)
plot_topic_account_stats(topic_account_stats, topic_col_name, output_folder_path)

# Clean Entities

In [ ]:
@report_done
def clean_entity_columns(df, entity_cols_lst):
    """Normalize entity columns to match TfidfVectorizer behavior.

    Converts each entity to a stripped, lowercase string for all columns
    listed in ``entity_cols_lst``.
    The function modifies ``df``.

    Args:
        df: Input DataFrame.
        entity_cols_lst: Columns containing lists of entities.

    Returns:
        None.
    """
    for col in entity_cols_lst:
        df[col] = df[col].apply(lambda lst: [str(e).strip().lower().replace(" ", "_") for e in lst])

In [ ]:
entity_cols_lst = ["hashtags", "account_mentions", "urls", "ner_entities"]
clean_entity_columns(df, entity_cols_lst)

# Entities Statistics

In [ ]:
@report_done
def entity_topic_counts(
    df: pd.DataFrame,
    entity_col: str,
    topic_col: str,
) -> pd.DataFrame:
    """Return entity-topic count matrix.

    Args:
        df: Input DataFrame.
        entity_col: Column with entity IDs (list-like).
        topic_col: Column with topic labels.

    Returns:
        DataFrame with entities as index, topics as columns, counts as values.
    """
    
    df_exploded = (
        df[[entity_col, topic_col]]
        .explode(entity_col)
        .dropna(subset=[entity_col, topic_col])
    )

    return pd.crosstab(
        index=df_exploded[entity_col],
        columns=df_exploded[topic_col],
        dropna=False,
    )


In [ ]:
def plot_entity_topic_counts(
    entity,
    entity_topic_counts,
    clustering_model: str = "",
    output_folder_path=None,
):
    """Plot how many times a given entity appears in each topic.

    Args:
        entity: Entity ID that exists in entity_topic_counts.index.
        entity_topic_counts: DataFrame from pd.crosstab (index=entity, columns=topic, values=counts).
        clustering_model: Optional label to annotate the plot.
        output_folder_path: If given (Path-like), saves PNG to this folder.
    """
    # Extract row for chosen entity (entity is in the index)
    counts = entity_topic_counts.loc[entity]  # Series: topic_id -> count

    # Topic columns
    topic_cols = list(entity_topic_counts.columns)


    # --- Plot ---
    plt.figure(figsize=(12, 5))
    plt.bar(topic_cols, counts.values)

    plt.title(f"Topic Distribution for Entity: '{entity}'")
    plt.xlabel("Topic ID")
    plt.ylabel("Count of occurrences")
    plt.xticks(topic_cols)
    plt.grid(axis="y")

    # Metadata note (top-right)
    plt.text(
        0.98, 0.92,
        f"clustering model: {clustering_model}",
        ha="right",
        va="top",
        transform=plt.gca().transAxes
    )

    plt.tight_layout()

    # Save if path provided
    if output_folder_path is not None:
        plot_path = output_folder_path / f"{entity}_topic_distribution.png"
        plt.savefig(plot_path, dpi=300)
        print(f"Saved figure to: {plot_path}")

    plt.show()
    plt.close()

In [ ]:
entity_col_name = "ner_entities"
topic_col = "BERTopic_topic"

entity_topic_counts_df = entity_topic_counts(df, entity_col_name, topic_col)
display(entity_topic_counts_df.head())

entity = entity_topic_counts_df.index[0]   # first entity
plot_entity_topic_counts(entity=entity, entity_topic_counts=entity_topic_counts_df, clustering_model=topic_col, output_folder_path=output_folder_path)

# Save to parquet

In [ ]:
# print dataset statistics
for col in df.columns:
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")

In [ ]:
save_df(df, output_folder_path, file_name)
write_report("Finished Running Successfully")
write_report("#######################################################")
write_report("#######################################################")